# SemEval Task 9 POLAR

### Subtask 3 - Manifestation Identification

### Hyper-parameter tuning - "microsoft/mdeberta-v3-base"

By: Kevin Mcmahon, Caleb Kumar

Note: This notebook was authored / ran in google colab to be able to easily accesss a GPU to speed up runtimes. This notebook ran using a Nvidia L4 GPU.

Mount Google Drive to allow access to files stored in the user's Drive. This command will prompt the user for authorization.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Saving the Data Directory**



In [3]:
base_dir = '/content/drive/MyDrive/SemEval2026/'
data_dir = base_dir + 'data/subtask3/'

### Import statements

In [4]:
!pip install optuna
!pip install -U transformers

import pandas as pd

from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import random
import math

import torch

from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
)
from torch.utils.data import Dataset
import wandb
from transformers import AutoConfig, AutoModelForSequenceClassification

import optuna
from optuna.samplers import TPESampler

import gc

### Importing / Formatting Data

In [5]:
base_dir = '/content/drive/MyDrive/SemEval2026/'
data_dir = base_dir + 'data/subtask3/'

from sklearn.model_selection import train_test_split

languages = ["eng","arb","deu"]

train_dfs = {}
dev_dfs = {}

for lang in languages:
    train_dfs[lang] = pd.read_csv(data_dir + f"train/{lang}.csv")
    train_dfs[lang]["language"] = lang
    dev_dfs[lang] = pd.read_csv(data_dir + f"dev/{lang}.csv")
    dev_dfs[lang]["language"] = lang

train_full = pd.concat([train_dfs[lang] for lang in languages], ignore_index=True)
dev_full = pd.concat([dev_dfs[lang] for lang in languages], ignore_index=True)

# 80/20 split for train/validation, preserving language distribution
train, validation = train_test_split(
    train_full,
    test_size=0.2,
    random_state=42,
    stratify=train_full["language"]
)

### Defining Dataset Object

In [6]:
# Dataset class (already multi-label friendly)
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # multi-label → float labels
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

### Defining Evaluation Metric

In [7]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics_multilabel(p):
    # p.predictions is a numpy array of logits: (num_examples, num_labels)
    logits = torch.tensor(p.predictions)
    probs = torch.sigmoid(logits).numpy()

    # Try a slightly lower threshold to avoid "all zeros" early on.
    # You can tune this later; 0.3–0.4 is often more sensible for imbalanced multilabel.
    threshold = 0.3
    preds = (probs >= threshold).astype(int)

    labels = p.label_ids

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    # This is *subset* accuracy (exact match over all 6 labels for each example)
    accuracy = accuracy_score(labels, preds)

    return {
        "f1_macro": f1_macro,
        "accuracy": accuracy,
    }

In [8]:
MODEL_NAME = "microsoft/mdeberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128

label_cols = [
    "vilification",
    "extreme_language",
    "stereotype",
    "invalidation",
    "lack_of_empathy",
    "dehumanization",
]

train_dataset = PolarizationDataset(
    train["text"].tolist(),
    train[label_cols].values.tolist(),
    tokenizer,
    max_length=MAX_LEN,
)

val_dataset = PolarizationDataset(
    validation["text"].tolist(),
    validation[label_cols].values.tolist(),
    tokenizer,
    max_length=MAX_LEN,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
 

### Defining Callback Metrics

This will allow us to evalaute the performance of the experiment so that we can make sure that the model is learning.

In [9]:
from transformers import TrainerCallback

class EpochMetricsCallback(TrainerCallback):
    def __init__(self, trainer, train_dataset, val_dataset, trial=None):
        super().__init__()
        self.trainer = trainer
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.trial = trial
        self.best_val_f1 = 0.0

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = state.epoch
        epoch_str = f"{epoch:.0f}" if epoch is not None and not math.isnan(epoch) else "?"

        print(f"\n===== Epoch {epoch_str} =====")

        # ---- Train metrics ----
        train_metrics = self.trainer.evaluate(eval_dataset=self.train_dataset)
        train_loss = train_metrics.get("eval_loss", float("nan"))
        train_f1 = train_metrics.get("eval_f1_macro", float("nan"))
        train_acc = train_metrics.get("eval_accuracy", float("nan"))
        print(
            "Train - "
            f"loss={train_loss:.4f}, "
            f"F1={train_f1:.4f}, "
            f"Acc={train_acc:.4f}"
        )

        # ---- Validation metrics ----
        val_metrics = self.trainer.evaluate(eval_dataset=self.val_dataset)
        val_loss = val_metrics.get("eval_loss", float("nan"))
        val_f1 = val_metrics.get("eval_f1_macro", 0.0)
        val_acc = val_metrics.get("eval_accuracy", float("nan"))
        print(
            "Val   - "
            f"loss={val_loss:.4f}, "
            f"F1={val_f1:.4f}, "
            f"Acc={val_acc:.4f}"
        )
        print("=========================\n")

        # Track best validation F1 for this trial
        if val_f1 > self.best_val_f1:
            self.best_val_f1 = val_f1

        # Report to Optuna + pruning
        if self.trial is not None:
            step = int(epoch) if epoch is not None and not math.isnan(epoch) else state.global_step
            self.trial.report(val_f1, step=step)
            if self.trial.should_prune():
                print(f"Pruning trial at epoch {epoch_str} with val F1={val_f1:.4f}")
                raise optuna.exceptions.TrialPruned()


### Optuna Objective

Optuna is a hyper-parameter tuning framework used to automate the process of hyper-parameter tuning while also doing it in a more efficient way using math rather than guess & check. I have previously used this in my final project for Deep Learning. The objective sets which parameters should be tuned, what the bounds for tuning the parameters are and what is the goal of the "sudy" - max Macro F1 in our case. Optuna will automatically suggest new parameters to test given the perfromance of the previously tested parameters. It will keep trying new "trials" for as long as you set unless it hits a time limit that you set... important when working in Google Colab.

In [10]:
import optuna
import gc
import torch
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

def build_model(dropout: float):
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_cols),
        problem_type="multi_label_classification",
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
    )
    return model

GLOBAL_SEED = 42  # pick any int you like and stick with it


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # HuggingFace helper (sets some internal RNGs)
    set_seed(seed)


# assumes:
# - GLOBAL_SEED is defined (e.g., GLOBAL_SEED = 42)
# - seed_everything(seed: int) is defined
# - build_model(dropout: float) is defined
# - compute_metrics_multilabel is defined
# - train_dataset, val_dataset, tokenizer, MODEL_NAME, label_cols are defined


def objective(trial: optuna.trial.Trial) -> float:
    trial_seed = GLOBAL_SEED + trial.number
    seed_everything(trial_seed)

    # Narrow LR around ~1e-5
    learning_rate = trial.suggest_float(
        "learning_rate",
        5e-6,    # lower
        2e-5,    # upper
        log=True,
    )

    # You’ve seen good behavior by 4–8 epochs
    num_train_epochs = trial.suggest_int("num_train_epochs", 4, 8)

    # If VRAM is fine, you can even fix this to 16
    per_device_train_batch_size = trial.suggest_categorical(
        "per_device_train_batch_size",
        [16],
    )

    # WD around 0.066
    weight_decay = trial.suggest_float("weight_decay", 0.03, 0.08)

    # Warmup around 0.05
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.02, 0.08)

    # Dropout around 0.15
    dropout = trial.suggest_float("dropout", 0.12, 0.22)

    model = build_model(dropout)
    training_args = TrainingArguments(
        output_dir="./subtask3_tmp",
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_train_batch_size,
        save_strategy="no",
        logging_steps=600,
        report_to="none",
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        metric_for_best_model="f1_macro",
        load_best_model_at_end=False,
        fp16=torch.cuda.is_available(),
        seed=trial_seed,
        data_seed=trial_seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_multilabel,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    epoch_cb = EpochMetricsCallback(
        trainer=trainer,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        trial=trial,
    )
    trainer.add_callback(epoch_cb)

    print("Trainer device:", trainer.args.device)

    try:
        trainer.train()
        eval_results = trainer.evaluate()
        print("Final eval metrics:", eval_results)
        f1 = max(epoch_cb.best_val_f1, eval_results["eval_f1_macro"])
    finally:
        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return f1

### Run Optuna "Study"

Above we defined the study, here we are actually going to run it.

Best hyper-parameters so you do't have to scroll through all of the logs.

Best F1: 0.5427844359604229
Best params: {'learning_rate': 1.9071114432274296e-05, 'num_train_epochs': 8, 'per_device_train_batch_size': 16, 'weight_decay': 0.07056562475456848, 'warmup_ratio': 0.035855957038224535, 'dropout': 0.17914280432053034}

In [11]:
study_name = f"study_{MODEL_NAME}"

sampler = TPESampler(seed=GLOBAL_SEED)

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
)

print(f"Starting study '{study_name}' with a 4-hour timeout...")
study.optimize(objective, n_trials=20, timeout=14400)

print("Best F1:", study.best_value)
print("Best params:", study.best_trial.params)


[I 2025-11-25 15:57:06,561] A new study created in memory with name: study_microsoft/mdeberta-v3-base


Starting study 'study_microsoft/mdeberta-v3-base' with a 4-hour timeout...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainer device: cuda:0


Step,Training Loss,Validation Loss,F1 Macro,Accuracy
462,No log,0.422055,0.475104,0.517857
600,0.497000,No Log,No Log,No Log
924,0.497000,0.412674,0.478620,0.522186
1200,0.408700,No Log,No Log,No Log
1386,0.408700,0.403326,0.516835,0.498377
1800,0.376900,No Log,No Log,No Log
1848,0.376900,0.409150,0.530518,0.511905
2310,0.376900,0.412109,0.535217,0.521104
2400,0.354900,No Log,No Log,No Log
2772,0.354900,0.423446,0.537653,0.519481


...

Trainer device: cuda:0


Step,Training Loss,Validation Loss,F1 Macro,Accuracy
462,No log,0.424841,0.445920,0.459416



===== Epoch 1 =====
Train - loss=0.4310, F1=0.4688, Acc=0.4390


[I 2025-11-25 17:31:54,939] Trial 19 pruned. 


Val   - loss=0.4248, F1=0.4459, Acc=0.4594

Pruning trial at epoch 1 with val F1=0.4459
Best F1: 0.5427844359604229
Best params: {'learning_rate': 1.9071114432274296e-05, 'num_train_epochs': 8, 'per_device_train_batch_size': 16, 'weight_decay': 0.07056562475456848, 'warmup_ratio': 0.035855957038224535, 'dropout': 0.17914280432053034}
